# Stacking SVR + XGBoost

Les modèles de base sont déjà entraînés : seules leurs prédictions sont nécessaires.
On entraîne une régression linéaire sur les prédictions OOF, puis on l’applique aux submissions test.
Tous les fichiers sont alignés par `id`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from IPython.display import display

# Fonctionne depuis Stacking, le dossier du projet ou la racine du dépôt.
ROOT = next(p.resolve() for p in [Path(".."), Path("."), Path("eth-aml-2026-project-1")]
            if (p / "sample.csv").exists() and (p / "Stacking").is_dir())
STACKING = ROOT / "Stacking"


In [2]:
def read_predictions(path, name):
    frame = pd.read_csv(path, index_col="id")
    assert frame.index.is_unique, f"IDs dupliqués : {path}"
    assert frame.shape[1] == 1, f"Une seule colonne de prédictions attendue : {path}"
    return frame.iloc[:, 0].rename(name)

svr_oof = read_predictions(STACKING / "oof_svr.csv", "svr")
xgb_oof = read_predictions(STACKING / "oof_xgb.csv", "xgb")
assert set(svr_oof.index) == set(xgb_oof.index), "Les IDs OOF diffèrent"
X_meta = pd.concat([svr_oof, xgb_oof], axis=1).loc[svr_oof.index]
y = pd.read_csv(ROOT / "y_train.csv", index_col="id")["y"].loc[X_meta.index]
folds = pd.read_csv(STACKING / "folds.csv", index_col="id")["fold"].loc[X_meta.index]
assert np.isfinite(X_meta.to_numpy()).all()
assert np.isfinite(y.to_numpy()).all()
assert folds.notna().all() and folds.nunique() >= 2
print(f"{len(X_meta)} observations OOF, {folds.nunique()} folds")
display(X_meta.head())


1152 observations OOF, 5 folds


,svr,xgb
id,,
0.0,73.797032,75.424614
1.0,54.568113,58.704491
2.0,67.637202,69.802628
4.0,71.562736,71.759781
5.0,66.504427,61.210045


## Validation exploratoire

Le méta-modèle est évalué sur les folds partagés : chaque prédiction est faite par une régression entraînée sur les autres folds.
La moyenne 50/50 est une référence sans entraînement.

**Limite :** les OOF sont déjà calculées. Les modèles de base ayant produit les OOF des autres folds peuvent avoir vu les cibles du fold de validation du méta-modèle. Cette évaluation ne remplace donc pas une validation imbriquée de toute la chaîne. Le prétraitement et le choix des hyperparamètres peuvent également introduire un optimisme s’ils ont utilisé toutes les données.


In [3]:
stack_oof = pd.Series(np.nan, index=X_meta.index, name="stacking")
for fold in sorted(folds.unique()):
    train, valid = folds != fold, folds == fold
    meta = LinearRegression().fit(X_meta.loc[train], y.loc[train])
    stack_oof.loc[valid] = meta.predict(X_meta.loc[valid])

predictions = {
    "SVR": X_meta["svr"],
    "XGBoost": X_meta["xgb"],
    "Moyenne 50/50": X_meta.mean(axis=1),
    "Stacking linéaire (CV exploratoire)": stack_oof,
}
metrics = pd.DataFrame({
    name: {"R2": r2_score(y, pred),
           "RMSE": np.sqrt(mean_squared_error(y, pred)),
           "MAE": mean_absolute_error(y, pred)}
    for name, pred in predictions.items()
}).T
display(metrics.round(5))


,R2,RMSE,MAE
SVR,0.69347,5.18535,3.90660
XGBoost,0.67872,5.30860,4.14465
Moyenne 50/50,0.71386,5.00989,3.82508
Stacking linéaire (CV exploratoire),0.71616,4.98969,3.79485


## Entraînement final et submissions

La régression finale utilise toutes les OOF. Les deux submissions de base doivent provenir de modèles entraînés sur toutes les données, avec les mêmes choix de modèle et de prétraitement que pour leurs OOF.
Les coefficients sont libres : ils ne sont pas contraints à être positifs ni à sommer à 1.


In [4]:
meta_model = LinearRegression().fit(X_meta, y)
display(pd.Series(meta_model.coef_, index=X_meta.columns, name="coefficient"))
print(f"Intercept : {meta_model.intercept_:.6f}")

svr_test = read_predictions(ROOT / "SVR" / "submission_svr.csv", "svr")
xgb_test = read_predictions(ROOT / "XGboost_ridge" / "submission_xgboost_knn.csv", "xgb")
sample = pd.read_csv(ROOT / "sample.csv")
assert sample["id"].is_unique
assert set(sample["id"]) == set(svr_test.index) == set(xgb_test.index), "IDs test différents"
X_test_meta = pd.concat([svr_test, xgb_test], axis=1).loc[sample["id"], X_meta.columns]
assert np.isfinite(X_test_meta.to_numpy()).all()

for filename, pred in {
    "submission_stacking.csv": meta_model.predict(X_test_meta),
    "submission_blend_50_50.csv": X_test_meta.mean(axis=1).to_numpy(),
}.items():
    submission = sample.copy()
    submission["y"] = pred
    assert np.isfinite(submission["y"]).all()
    destination = STACKING / filename
    submission.to_csv(destination, index=False)
    print(f"Enregistré : {destination.name} ({len(submission)} lignes)")

display(pd.read_csv(STACKING / "submission_stacking.csv").head())


svr    0.600401
xgb    0.469515
Name: coefficient, dtype: float64

Intercept : -4.896383
Enregistré : submission_stacking.csv (776 lignes)
Enregistré : submission_blend_50_50.csv (776 lignes)


,id,y
0,0.0,61.061919
1,1.0,74.070571
2,2.0,69.268984
3,3.0,75.219934
4,4.0,75.839797
